# 🛒 ShelfSense — YOLOv8 Product Counter Training

Fine-tune YOLOv8n on **SKU-110K** using **chunked training** (survives disconnects).

## Strategy
- Train in **20-epoch chunks** saved directly to Google Drive
- If session dies → reconnect → resume from last checkpoint
- Repeat until 50 epochs total

## Steps
1. **Runtime → Change runtime type → T4 GPU**
2. Run Cell 1 (GPU check)
3. Run Cell 2 (Mount Drive)
4. Run Cell 3 (Train — first chunk OR resume)
5. If disconnected → reconnect → run Cells 1, 2, 3 again (it auto-resumes)
6. When done → run Cells 4, 5, 6

---

## 1️⃣ Check GPU & Install Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi

# Install ultralytics
!pip install -q ultralytics

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ NO GPU — go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Mount Google Drive

**All training output goes directly to Drive** — survives disconnects and quota limits.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/ShelfSense'
DRIVE_RUNS = f'{DRIVE_PROJECT}/runs'
WEIGHTS_DIR = f'{DRIVE_RUNS}/shelfsense/weights'
os.makedirs(DRIVE_PROJECT, exist_ok=True)

# Check if a previous checkpoint exists
last_pt = f'{WEIGHTS_DIR}/last.pt'
if os.path.exists(last_pt):
    size_mb = os.path.getsize(last_pt) / (1024*1024)
    print(f"\n🔄 FOUND CHECKPOINT: {last_pt} ({size_mb:.1f} MB)")
    print("   Cell 3 will RESUME from this checkpoint automatically!")
else:
    print(f"\n🆕 No checkpoint found — Cell 3 will start fresh training.")

print(f"\n✅ Drive mounted. Output dir: {DRIVE_PROJECT}")

## 3️⃣ Train (Auto-Resumes from Checkpoint)

This cell is smart:
- **First run**: starts fresh, trains 20 epochs
- **After disconnect**: detects the checkpoint on Drive and resumes where it left off
- **Keep re-running** until you hit 50 total epochs

⏱️ Each 20-epoch chunk takes ~2.5-3 hours on T4

In [ ]:
from ultralytics import YOLO
import os

WEIGHTS_DIR = '/content/drive/MyDrive/ShelfSense/runs/shelfsense/weights'
last_pt = f'{WEIGHTS_DIR}/last.pt'

# ── Auto-detect: resume or fresh start ──
if os.path.exists(last_pt):
    print("🔄 RESUMING from last checkpoint on Drive...")
    print(f"   Loading: {last_pt}")
    model = YOLO(last_pt)
    results = model.train(
        resume=True,        # ← This tells YOLO to continue from where it stopped
    )
else:
    print("🆕 Starting FRESH training...")
    model = YOLO('yolov8n.pt')
    results = model.train(
        data='SKU-110K',
        epochs=50,               # Total target epochs
        batch=16,
        imgsz=640,
        device=0,
        project='/content/drive/MyDrive/ShelfSense/runs',  # ← SAVES TO DRIVE
        name='shelfsense',
        exist_ok=True,
        # Augmentation
        mosaic=1.0,
        mixup=0.1,
        degrees=5.0,
        translate=0.1,
        scale=0.3,
        fliplr=0.5,
        flipud=0.0,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.4,
        # Optimization
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=3,
        cos_lr=True,
        patience=10,
        # Output — save frequently!
        save=True,
        save_period=5,           # ← Checkpoint every 5 epochs (was 10)
        plots=True,
        verbose=True,
    )

print("\n" + "="*60)
print("✅ TRAINING COMPLETE (or paused)!")
print("="*60)
print(f"\nCheckpoint saved to: {WEIGHTS_DIR}/")
print("If you need more epochs, just re-run this cell.")

## 4️⃣ Evaluate the Trained Model

In [ ]:
from ultralytics import YOLO

WEIGHTS_DIR = '/content/drive/MyDrive/ShelfSense/runs/shelfsense/weights'

# Use best.pt if it exists, otherwise use last.pt
import os
best_pt = f'{WEIGHTS_DIR}/best.pt'
last_pt = f'{WEIGHTS_DIR}/last.pt'
weights = best_pt if os.path.exists(best_pt) else last_pt

print(f"Evaluating: {weights}")
best_model = YOLO(weights)
metrics = best_model.val(data='SKU-110K', device=0)

print("\n" + "="*60)
print("📊 EVALUATION RESULTS")
print("="*60)
print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print("="*60)

## 5️⃣ Visualize Training Results

In [ ]:
from IPython.display import Image, display
import os

results_dir = '/content/drive/MyDrive/ShelfSense/runs/shelfsense'

plots = ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png']
for plot_name in plots:
    plot_path = os.path.join(results_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"\n📈 {plot_name}")
        display(Image(filename=plot_path, width=800))

val_preds = os.path.join(results_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_preds):
    print("\n🔍 Sample Predictions (Validation Batch):")
    display(Image(filename=val_preds, width=900))

## 6️⃣ Export Final Model

Copy the best weights to a clean filename for download.

In [ ]:
import shutil
import os

DRIVE_PROJECT = '/content/drive/MyDrive/ShelfSense'
WEIGHTS_DIR = f'{DRIVE_PROJECT}/runs/shelfsense/weights'

best_pt = f'{WEIGHTS_DIR}/best.pt'
last_pt = f'{WEIGHTS_DIR}/last.pt'
src = best_pt if os.path.exists(best_pt) else last_pt
dst = f'{DRIVE_PROJECT}/yolov8n-shelfsense.pt'

shutil.copy2(src, dst)

# Also copy training plots
results_dir = f'{DRIVE_PROJECT}/runs/shelfsense'
for plot in ['results.png', 'confusion_matrix.png']:
    plot_src = f'{results_dir}/{plot}'
    if os.path.exists(plot_src):
        shutil.copy2(plot_src, f'{DRIVE_PROJECT}/{plot}')

print("\n" + "="*60)
print("✅ MODEL EXPORTED TO GOOGLE DRIVE")
print("="*60)
print(f"\n📦 Model: {dst}")
print(f"📁 All files: {DRIVE_PROJECT}/")
print("\n📋 Next Steps:")
print("  1. Download 'yolov8n-shelfsense.pt' from your Google Drive")
print("  2. Place it in your ShelfSense/ project root")
print("  3. We'll integrate it into the pipeline")
print("\n🎉 You're done with the training phase!")